# Init

In [ ]:
# init
from importlib import reload
from pathlib import Path
import os
import sys
import pandas as pd

sys.path.append('..')
sys.path.append("C:/Users/gonta/D/software developer/packages")
import toolsGeneral.main as tgm
import toolsPandas.helpers as tph

def reloadPckg():
    reload(tgm)
    reload(tph)

# Import

In [1]:
# import
dataDir = Path(os.path.join(os.getcwd(), '..', 'data/normalized'))
rawFiles = [f for f in dataDir.rglob('*.pkl')]
print(len(rawFiles))

df_by_cntr = {
    file.parent.name:tgm.load(str(file))
    for file in rawFiles
}
print(len(df_by_cntr))

64
64


In [2]:
df_all = pd.concat(df_by_cntr.values(), ignore_index=True)
print(len(df_all))

118738


In [3]:
all_id_tuples = df_all[['id','tags.parent_id','tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
print(len(all_id_tuples))

118738


# apply fixes

In [4]:
basic_to_delete = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm basic test", "basic_to_delete.pkl"))
print(len(basic_to_delete))

dups_tuples_to_delete = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm duplicates test", "dups_tuples_to_delete.pkl"))
print(len(dups_tuples_to_delete))

tuples_to_delete = tgm.load(os.path.join(os.getcwd(), "../data/tests results/osm first level center test", "tuples_to_delete.pkl"))
print(len(tuples_to_delete))

865
120
87


In [5]:
to_delete = list(set(basic_to_delete) | set(dups_tuples_to_delete) | set(tuples_to_delete))
print(len(to_delete))

1041


In [27]:
df_by_cntr_fixed = {}
for k,df in df_by_cntr.items():
    df_by_cntr_fixed[k] = df[~df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(to_delete)]
print(len(df_by_cntr_fixed))

64


In [29]:
df_all_fixed =  pd.concat(df_by_cntr_fixed.values(), ignore_index=True)
print(len(df_all_fixed))

117697


In [32]:
print(len(df_all))
print(len(df_all)-len(to_delete))
print(len(df_all_fixed))

118738
117697
117697


# save

In [33]:
for k,df in df_by_cntr_fixed.items():
    path = os.path.join(os.getcwd(),f'../data/fixed/{k}/{k}_fixed.pkl')
    tgm.dump(path, df)